# Авторское решение: логистическая регрессия и завершение курса

Блокнот показывает полный ход: признаки, сравнение параметров `C` и `class_weight`, подбор порога по F1 и финальный файл.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_DIR = Path("data")


In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")

target = "will_finish"
id_col = "id"


In [ ]:
def add_features(df):
    df = df.copy()
    df["activity_total"] = df["practice_hours_week"] + 0.5 * df["club_visits"]
    df["score_per_missed_class"] = df["last_score"] / (df["missed_classes"] + 1)
    df["teacher_messages_per_week"] = df["messages_to_teacher"] / 4
    df["is_many_missed"] = (df["missed_classes"] >= 4).astype(int)
    df["strong_homework"] = (df["homework_rate"] >= 0.8).astype(int)
    df["low_last_score"] = (df["last_score"] < 50).astype(int)
    return df


train_fe = add_features(train)
test_fe = add_features(test)

features = [c for c in train_fe.columns if c not in [target, id_col]]
X = train_fe[features]
y = train_fe[target]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)


In [ ]:
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocess(X_part):
    numeric_features = X_part.select_dtypes(include=np.number).columns.tolist()
    categorical_features = X_part.select_dtypes(exclude=np.number).columns.tolist()
    return ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", make_ohe()),
                    ]
                ),
                categorical_features,
            ),
        ]
    )


def best_threshold_for_f1(y_true, proba):
    thresholds = np.linspace(0.15, 0.85, 71)
    scores = []
    for threshold in thresholds:
        pred = (proba >= threshold).astype(int)
        scores.append(f1_score(y_true, pred, zero_division=0))
    best_index = int(np.argmax(scores))
    return float(thresholds[best_index]), float(scores[best_index])


def evaluate_model(name, estimator):
    model = Pipeline(
        steps=[
            ("preprocess", make_preprocess(X_train)),
            ("model", estimator),
        ]
    )
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_val)[:, 1]
    threshold, best_f1 = best_threshold_for_f1(y_val, proba)
    pred = (proba >= threshold).astype(int)
    return {
        "name": name,
        "threshold": threshold,
        "F1": best_f1,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "accuracy": accuracy_score(y_val, pred),
        "ROC_AUC": roc_auc_score(y_val, proba),
        "model": model,
    }


In [ ]:
candidates = []
for C in [0.3, 1.0, 3.0, 10.0]:
    candidates.append((f"LogReg C={C}", LogisticRegression(max_iter=1000, C=C)))
    candidates.append(
        (
            f"LogReg C={C}, balanced",
            LogisticRegression(max_iter=1000, C=C, class_weight="balanced"),
        )
    )

rows = []
models = {}
estimators_by_name = {name: estimator for name, estimator in candidates}
for name, estimator in candidates:
    result = evaluate_model(name, estimator)
    models[name] = result.pop("model")
    rows.append(result)

leaderboard = pd.DataFrame(rows).sort_values("F1", ascending=False)
leaderboard


In [ ]:
best_row = leaderboard.iloc[0]
best_name = best_row["name"]
best_threshold = float(best_row["threshold"])

print("Лучшая модель:", best_name)
print("Порог:", round(best_threshold, 3))


In [ ]:
final_model = Pipeline(
    steps=[
        ("preprocess", make_preprocess(X)),
        ("model", clone(estimators_by_name[best_name])),
    ]
)

final_model.fit(X, y)
test_proba = final_model.predict_proba(test_fe[features])[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

submission = pd.DataFrame(
    {
        "id": test_fe[id_col],
        "probability": test_proba.round(5),
        "will_finish": test_pred,
    }
)

submission.to_csv("author_submission.csv", index=False)
submission.head()
